# LIME — "Why Should I Trust You?": Explaining the Predictions of Any Classifier (2016)

_Papers · Meta-learning, Bayesian & Robustness_

**Explain one prediction of any black-box model by fitting a simple linear model that is faithful only nearby.**

---

This notebook is a practice scaffold for the **LIME — "Why Should I Trust You?": Explaining the Predictions of Any Classifier (2016)** lesson. We build LIME one step at a time: a tiny worked example, then a black-box classifier, then sample-label-weight-fit the local surrogate, and finally check that the explanation is faithful near the point but not far. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — NumPy

### Step 1 — Fit a 1-feature local surrogate by hand

LIME explains a prediction by fitting a simple linear model to the black box's outputs *near* the point of interest, weighting nearby samples more. Here the black box is `f(z) = z²` and we explain it at `x0 = 2`. We weight three samples by a proximity kernel `pi`, then solve the **weighted normal equations** `(XᵀWX)c = XᵀWf`. The recovered slope `c1` should match the true derivative `df/dz = 2x = 4` at `x0 = 2`.

In [ ]:
# Worked example: black box f(z)=z^2, explain at x=2.
x0 = 2.0
sigma = 1.0
z = np.array([1.0, 2.0, 3.0])
f = z**2                                       # black-box outputs: [1, 4, 9]

pi = np.exp(-((x0 - z)**2) / sigma**2)         # proximity weights (closer z -> larger weight)
X = np.column_stack([np.ones_like(z), z])      # design matrix [1, z]
W = np.diag(pi)                                 # diagonal weight matrix

# Weighted normal equations: (X^T W X) c = X^T W f
c = np.linalg.solve(X.T @ W @ X, X.T @ W @ f)

print("worked example: pi =", np.round(pi, 4))
print("  surrogate coeffs (c0 intercept, c1 slope) =", np.round(c, 4))
print("  c1 =", round(c[1], 4), " vs true df/dz at x=2 = 2*2 =", 2*x0)
# worked example: pi = [0.3679 1.     0.3679]
#   surrogate coeffs (c0 intercept, c1 slope) = [-3.5761  4.    ]
#   c1 = 4.0  vs true df/dz at x=2 = 2*2 = 4.0

### Step 2 — Define the black box and pick the point to explain

LIME is **model-agnostic**: it only ever *calls* the model, never inspects it. Our black box is a closed-form nonlinear classifier mapping `R²` to a probability via a sigmoid. We pick one input `x` — the single prediction we want to explain.

In [ ]:
# The BLACK BOX: a closed-form nonlinear classifier (numpy only).
# We only CALL it -> model-agnostic. Maps R^2 -> probability [0,1].
def black_box(Z):
    Z = np.atleast_2d(Z)
    s = 3.0*Z[:,0] + 1.5*Z[:,0]*Z[:,1] - 1.2*(Z[:,1]**2)
    return 1.0 / (1.0 + np.exp(-s))            # sigmoid

x = np.array([0.5, 0.2])                       # the ONE point we explain
print("explain prediction at x =", x, " f(x) =", round(float(black_box(x)[0]), 4))

### Step 3 — Sample, label, weight, and fit the local surrogate

This is the heart of LIME. We (1) perturb `x` to get a cloud of nearby samples, (2) *label* them by calling the black box, (3) *weight* each by a proximity kernel `pi_x` so closer samples count more, then (4) solve a **weighted least squares** for the linear surrogate. The surrogate's two slopes are the local feature attributions.

In [ ]:
# LIME: sample -> label -> weight by proximity -> weighted least squares.
rng = np.random.default_rng(0)
N = 500
sig = 0.5

Zs = x + rng.normal(0, sig, size=(N, 2))       # 2. perturbed samples around x
y  = black_box(Zs)                             # 3. label with the black box

D   = np.linalg.norm(Zs - x, axis=1)           # L2 distance from x to each sample
pix = np.exp(-(D**2) / sig**2)                  # 4. proximity kernel pi_x(z)

Xd = np.column_stack([np.ones(N), Zs])         # design matrix [1, z0, z1]
A  = (Xd * pix[:, None]).T @ Xd                # X^T W X
b  = (Xd * pix[:, None]).T @ y                 # X^T W y
coef = np.linalg.solve(A, b)                   # weighted least squares

print("surrogate coeffs [intercept, w_feat0, w_feat1] =", np.round(coef, 4))
print("  local attribution: feature0 =", round(coef[1], 4),
      " feature1 =", round(coef[2], 4))

### Step 4 — Check faithfulness and compare to the true gradient

A LIME explanation should be faithful *near* `x` and is allowed to be wrong far away — that is the whole point of a *local* surrogate. We measure the surrogate-vs-black-box RMSE in a near band and a far band (far should be much worse), then confirm LIME's dominant feature matches the black box's true local gradient (finite-differenced).

In [ ]:
# FAITHFULNESS: local fit matches near x, not far. (RMSE)
def surrogate(Z):
    Z = np.atleast_2d(Z)
    return coef[0] + Z @ coef[1:]

def rmse(mask):
    return float(np.sqrt(np.mean((black_box(Zs[mask]) - surrogate(Zs[mask]))**2)))

near = D < 0.4
far  = D > 1.0
print("RMSE(surrogate vs black box)  NEAR x (D<0.4):", round(rmse(near), 4), " n=", int(near.sum()))
print("RMSE(surrogate vs black box)  FAR   (D>1.0):", round(rmse(far),  4), " n=", int(far.sum()))

# Did LIME recover the locally-important feature? Compare to true gradient.
eps = 1e-3
g0 = (black_box(x+[eps,0])[0] - black_box(x-[eps,0])[0]) / (2*eps)
g1 = (black_box(x+[0,eps])[0] - black_box(x-[0,eps])[0]) / (2*eps)
print("true local gradient: d/dfeat0 =", round(float(g0),4), " d/dfeat1 =", round(float(g1),4))
print("LIME's dominant feature:", "feature0" if abs(coef[1])>abs(coef[2]) else "feature1")

# --- typical output (numpy default_rng(0), CPU) ---
# explain prediction at x = [0.5 0.2]  f(x) = 0.8323
# surrogate coeffs [intercept, w_feat0, w_feat1] = [0.5342 0.4866 0.0129]
#   local attribution: feature0 = 0.4866  feature1 = 0.0129
# RMSE ... NEAR x (D<0.4): 0.0321  n= 146
# RMSE ... FAR   (D>1.0): 0.2804  n= 62      <- ~9x worse far away
# true local gradient: d/dfeat0 = 0.4606  d/dfeat1 = 0.0377
# LIME's dominant feature: feature0
# (Our small run, not the paper's reported numbers.)

## Visualize the data & results

_How faithful is LIME's local linear surrogate to the black box as we move away from the explained point?_

### Step 5 — Rebuild the black box, surrogate, and sample cloud

The visualization cell runs independently, so it re-defines the black box and re-fits the same weighted-least-squares surrogate around `x`. This is the identical sample → label → weight → fit pipeline from Step 3, condensed so the plotting step below has everything it needs.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)

# black box: closed-form nonlinear classifier, R^2 -> probability (numpy only)
def black_box(Z):
    Z = np.atleast_2d(Z)
    s = 3.0*Z[:,0] + 1.5*Z[:,0]*Z[:,1] - 1.2*(Z[:,1]**2)
    return 1.0/(1.0+np.exp(-s))

x   = np.array([0.5, 0.2])
N   = 500
sig = 0.5

Zs = x + rng.normal(0, sig, size=(N, 2))        # perturb around x
y  = black_box(Zs)                              # label with black box

D   = np.linalg.norm(Zs - x, axis=1)            # L2 distance
pix = np.exp(-(D**2)/sig**2)                     # proximity kernel pi_x

Xd = np.column_stack([np.ones(N), Zs])          # [1, z0, z1]
coef = np.linalg.solve((Xd*pix[:,None]).T @ Xd, (Xd*pix[:,None]).T @ y)   # weighted LS
print("surrogate coeffs [intercept, w0, w1]:", np.round(coef, 4))

def surrogate(Z):
    Z = np.atleast_2d(Z)
    return coef[0] + Z @ coef[1:]

### Step 6 — Watch faithfulness decay with distance

Here we bin the samples into distance bands from `x` and report the surrogate-vs-black-box RMSE in each. The error stays small in the innermost bands and grows steadily outward — a concrete picture of LIME's core promise: the explanation is faithful *near* `x` and is not meant to describe the model's global behavior.

In [ ]:
# RMSE of surrogate vs black box in distance bands -> faithfulness decays with distance
bands = [(0.0,0.25),(0.25,0.5),(0.5,0.75),(0.75,1.0),(1.0,1.5),(1.5,3.0)]
for lo, hi in bands:
    m = (D >= lo) & (D < hi)
    if m.sum() > 2:
        r = np.sqrt(np.mean((black_box(Zs[m]) - surrogate(Zs[m]))**2))
        center = (lo + hi) / 2
        print("band [%.2f,%.2f) center=%.3f  RMSE=%.4f  n=%d" % (lo, hi, center, r, int(m.sum())))
# surrogate coeffs [intercept, w0, w1]: [0.5342 0.4866 0.0129]
# band [0.00,0.25) center=0.125  RMSE=0.0428  n=54
# band [0.25,0.50) center=0.375  RMSE=0.0275  n=155
# band [0.50,0.75) center=0.625  RMSE=0.0844  n=146
# band [0.75,1.00) center=0.875  RMSE=0.1683  n=83
# band [1.00,1.50) center=1.250  RMSE=0.2567  n=58
# band [1.50,3.00) center=2.250  RMSE=0.5123  n=4
# Faithful near x, not far. Our small run, not the paper's number.

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The weighting ablation. You have a working LIME with proximity weights $\pi_x(z)$. Drop them:
            set every sample's weight to $1$ (an ordinary, unweighted least squares fit of the linear model to
            the black box outputs), everything else identical. What did you just remove, and what do you expect
            to happen to the explanation's local faithfulness?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Locate the weight matrix $W = \mathrm{diag}(\pi_x(z_i))$ in the weighted normal equations $(X^\top W X)c = X^\top W y$. Replace $W$ with the identity (all ones). — _With $W=I$ every sample counts equally, so the fit is no longer pinned to the neighborhood of $x$ &mdash; far samples drag the line just as much as near ones._
- Identify what is gone: the locality. The objective is no longer Equation 2 (proximity-weighted); it is a plain global least squares over the whole sample cloud. — _The proximity kernel $\pi_x$ is exactly the term that makes the surrogate local. Remove it and LIME stops being local._
- Refit and compare near-vs-far root-mean-square error: expect the near-x error to rise (worse local fit) while the far error may fall (the line now compromises across all regions). — _An unweighted line minimizes average error everywhere, so it trades away accuracy at $x$ to fit the bulk of the cloud._

**Answer:** You removed the proximity weighting &mdash; the heart of LIME's locality. With every
                 weight $1$, the fit becomes a global least squares over the whole perturbed cloud, so far
                 samples pull the line away from $x$. In our run, dropping the weights raises the root-mean-square
                 error near $x$ (the explanation gets less faithful exactly where it should be most
                 faithful) and can lower it far away. The lesson: $\pi_x$ is what makes the explanation a local
                 one; without it you are describing the average behavior, not the behavior at $x$.

</details>

**Problem 2.** Near our point, the black box's output depends almost entirely on feature 0; feature 1
            barely changes it there. Before fitting, predict the two surrogate coefficients $(c_1, c_2)$. After
            fitting we got $(c_1, c_2) \approx (0.49, 0.01)$ while the true local gradient of the black box was
            $(0.46, 0.04)$. Do these agree, and what does that tell you about what LIME's coefficients mean?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Reason from the setup: if moving feature 0 changes $f$ a lot locally and feature 1 hardly moves it, the local linear model should have a large slope on feature 0 and a near-zero slope on feature 1. — _A linear surrogate's coefficient is the local sensitivity of the output to that feature; small local sensitivity gives a small coefficient._
- Compare the fitted coefficients to the black box's true local gradient (its partial derivatives): $(0.49, 0.01)$ vs $(0.46, 0.04)$. Both say feature 0 dominates. — _LIME's weighted fit recovers the local linear behavior, so its coefficients track the black box's local gradient direction._
- Conclude that LIME identifies the locally-important feature (feature 0) and assigns the unimportant one (feature 1) a near-zero attribution. — _That is exactly the goal: a human-readable, sparse statement of which features drove this prediction._

**Answer:** They agree: the surrogate gives feature 0 a large weight ($\approx 0.49$) and
                 feature 1 a near-zero weight ($\approx 0.01$), matching the black box's true local
                 gradient $(0.46, 0.04)$. So LIME's coefficients are local feature attributions &mdash;
                 they recover which features the black box is actually sensitive to at this point. LIME
                 correctly singles out feature 0 as the locally-important one and (correctly) says
                 feature 1 did not matter here. The same model could rely on feature 1 elsewhere; the
                 explanation is only a claim about the neighborhood of $x$.

</details>

**Problem 3.** In the worked example you fit $g(z)=c_0+c_1 z$ to $f(z)=z^2$ at $x=2$ with samples $z=[1,2,3]$,
            weights $\pi=[0.3679,1,0.3679]$ ($\sigma=1$), getting $c=(-3.576, 4.000)$. Suppose you widen the
            kernel to $\sigma=3$. Recompute the weights, and say (qualitatively) what happens to the fitted
            slope and why.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recompute the proximity weights with $\sigma=3$: $\pi(1)=\exp(-1/9)\approx 0.8948$, $\pi(2)=\exp(0)=1$, $\pi(3)=\exp(-1/9)\approx 0.8948$. — _A larger bandwidth $\sigma$ raises the weight on the far samples ($0.3679 \to 0.8948$), so the near and far samples now count almost equally._
- Note the fit still passes data symmetric about $z=2$, so the weighted slope stays near the central slope of $z^2$ between $z=1$ and $z=3$, which is $\tfrac{9-1}{3-1}=4$. — _These three symmetric points give a slope of $4$ regardless of how the symmetric weights are set; the symmetry pins it._
- Reason about a NON-symmetric sample set or a finer curve: with a wide $\sigma$ the far points dominate more, so the surrogate would describe a broader, less-local region &mdash; the slope would drift toward the average behavior over that wider band, away from the true local derivative. — _Bandwidth controls how local the explanation is; widen it and you explain a bigger neighborhood, losing fidelity right at $x$._

**Answer:** With $\sigma=3$ the weights become $\pi \approx [0.8948, 1, 0.8948]$ &mdash; the far samples
                 now count almost as much as the central one. For these three symmetric points the slope happens
                 to stay $4$ (symmetry pins it). But the meaning changed: a wider $\sigma$ explains a
                 broader region, so in general the surrogate drifts from the true local derivative toward
                 the average behavior over a larger neighborhood. Bandwidth $\sigma$ is the knob for "how local"
                 &mdash; small for a tight, faithful-at-$x$ explanation; large for a coarser, more global one.

</details>